<a href="https://colab.research.google.com/github/onklover/wordremaster/blob/main/%EB%8F%99%EC%95%84%EB%A6%AC%ED%95%99%EC%8A%B5_%ED%85%8C%EC%8A%A4%ED%8A%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 외부 데이터셋 다운로드 링크 (이 부분만 원하는 데이터셋 링크로 변경하세요)
DATASET_URL = "https://www.dropbox.com/scl/fi/x9blrunioz9s348tqs60s/RockPaperScissors_Data.zip?rlkey=hj8f06qpkuz2k7vfq8534149b&st=0xhdimpc&dl=1"

# 2. 필수 라이브러리 설치
!pip install PyYAML ultralytics gradio

import os
import zipfile
import yaml
import glob
import cv2
from ultralytics import YOLO

# 3. 데이터셋 다운로드 및 압축 해제
!rm -rf /content/custom_dataset
!rm -f /content/custom_dataset.zip

print("\n--- [1/6] 데이터셋 다운로드 및 압축 해제 ---")
!wget -O /content/custom_dataset.zip {DATASET_URL}

os.makedirs('/content/custom_dataset', exist_ok=True)
with zipfile.ZipFile('/content/custom_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/custom_dataset/')
print("데이터셋 준비 완료!")

In [ ]:
# 4. YAML 파일 메타데이터 파싱 및 절대 경로 변환
print("\n--- [2/6] 메타데이터 분석 및 절대 경로 설정 ---")
yaml_files = glob.glob('/content/custom_dataset/**/*.yaml', recursive=True)
if not yaml_files:
    raise FileNotFoundError("압축 해제된 폴더 내에서 yaml 파일을 찾을 수 없습니다.")

# 오류 수정: glob은 리스트를 반환하므로 첫 번째 항목[0]을 선택
original_yaml_path = yaml_files[0]
with open(original_yaml_path, 'r') as f:
    data = yaml.safe_load(f)

base_dir = os.path.dirname(original_yaml_path)

# Train 경로 설정
train_img_path = os.path.join(base_dir, 'train', 'images')
data['train'] = train_img_path if os.path.exists(train_img_path) else os.path.join(base_dir, 'train')

# Valid 경로 설정
if os.path.exists(os.path.join(base_dir, 'valid', 'images')):
    data['val'] = os.path.join(base_dir, 'valid', 'images')
elif os.path.exists(os.path.join(base_dir, 'val', 'images')):
    data['val'] = os.path.join(base_dir, 'val', 'images')
else:
    data['val'] = os.path.join(base_dir, 'valid') if os.path.exists(os.path.join(base_dir, 'valid')) else os.path.join(base_dir, 'val')

# Test 경로 설정
test_img_path = os.path.join(base_dir, 'test', 'images')
if os.path.exists(test_img_path):
    data['test'] = test_img_path
elif os.path.exists(os.path.join(base_dir, 'test')):
    data['test'] = os.path.join(base_dir, 'test')

custom_yaml_path = '/content/custom_dataset/custom_data.yaml'
with open(custom_yaml_path, 'w') as f:
    yaml.dump(data, f)

print(f"새로운 YAML 파일 생성 완료: {custom_yaml_path}")
print(f"클래스 개수: {data.get('nc')}, 클래스명: {data.get('names')}")

In [ ]:
# 5. 모델 훈련
print("\n--- [3/6] 객체 탐지 모델 학습 시작 ---")
model = YOLO('yolo11n.pt')
results = model.train(
    data=custom_yaml_path,
    epochs=10,
    patience=5,
    imgsz=640,
    device=0
)

In [ ]:
# 6. 테스트 데이터 일괄 추론
print("\n--- [4/6] 테스트 데이터 일괄 추론 ---")
test_dir = data.get('test', '')
if test_dir and os.path.exists(test_dir):
    model(source=test_dir, save=True)
else:
    print("테스트 데이터가 없어 추론을 건너뜁니다.")

# 7. 추론 결과 자동 압축
print("\n--- [5/6] 추론 결과 이미지 압축 ---")
predict_dirs = glob.glob('/content/runs/detect/predict*')
if predict_dirs:
    latest_predict_dir = max(predict_dirs, key=os.path.getmtime)
    detected_images = glob.glob(os.path.join(latest_predict_dir, '*'))

    os.makedirs('/content/detected_result/', exist_ok=True)
    zip_path = '/content/detected_result/detected_images.zip'

    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for img_path in detected_images:
            zipf.write(img_path, os.path.basename(img_path))
    print(f"추론 결과가 성공적으로 압축되었습니다: {zip_path}")

In [ ]:
import gradio as gr

# 8. 동적 Gradio 웹 UI 배포
print("\n--- [6/6] 인터랙티브 웹 앱 배포 ---")
best_model = YOLO('/content/runs/detect/train/weights/best.pt')
class_names_str = ", ".join(best_model.names.values())

def predict(image):
    # 오류 수정: predict는 리스트를 반환하므로 [0]으로 첫 번째 결과 인덱싱
    res = best_model.predict(source=image, conf=0.25)
    annotated_image = res[0].plot()
    return cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)

iface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="이미지 업로드"),
    outputs=gr.Image(type="numpy", label="객체 탐지 결과"),
    title="범용 AI 객체 탐지기",
    description=f"이 모델은 다음 객체들을 탐지하도록 맞춤 학습되었습니다:\n**[{class_names_str}]**"
)

# 퍼블릭 링크를 통해 외부 접속 허용
iface.launch(share=True)